# Modulation Recognition: Handcrafted Features vs CNN (BPSK, QPSK, 16-QAM, 64-QAM)

**Assignment:** Chapter 3, option 2 (modulation recognition) - a simple project that covers every item the rubric asks for.

| Rubric item | Where it is covered |
|---|---|
| Scenario and signal model | Section 1 |
| Data generation (SNR, fading, CFO) | Section 1 |
| Features | Section 2 (handcrafted cumulants) and Section 3 (constellation image for the CNN) |
| Baseline | Cumulant features + random forest (Section 2) |
| ML method | Small CNN on constellation images (Section 3) |
| Metrics | Accuracy vs SNR, confusion matrices, accuracy vs CFO (Sections 5-6) |
| Leakage prevention | Split by whole **burst** (Section 4) and the random-split demo (Section 7) |
| Results and limitations | Sections 5-8 |
| Reproducible code | Fixed seeds, all parameters in one cell |

Run all cells in order (Runtime > Run all). A GPU is not required, it takes a few minutes on CPU.

In [ ]:
import time, copy, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

# ---------------- Experiment configuration (all parameters in one place) ----------------
SEEDS        = [0, 1, 2]                    # whole experiment is repeated for each seed
MODS         = ["BPSK", "QPSK", "16-QAM", "64-QAM"]
N_SYM        = 128                          # symbols per sample (one window)
WIN_PER_BURST = 4                           # windows cut from the same burst (they share channel, phase, CFO)
SNR_TRAIN    = np.arange(-5, 21, 5)         # training SNRs (dB): -5 ... 20
SNR_TEST     = np.arange(-10, 31, 5)        # test SNRs (dB): includes values lower and higher than training
BURSTS_PER_CELL = 150                       # bursts per (modulation, SNR, channel) in the training pool
TEST_BURSTS_PER_CELL = 60                   # bursts per cell in each fresh test set
TRAIN_CHANNELS = ["awgn", "rayleigh"]       # channels seen during training
RICIAN_K_DB  = 6.0                          # unseen channel used only for testing
CFO_TEST     = [0.0, 0.0005, 0.001, 0.002, 0.005]   # max |CFO| in cycles/symbol (unseen, test only)
CFO_TEST_SNR = 15                           # dB used for the CFO sweep
IMG, IMG_RANGE = 32, 2.5                    # constellation image: 32x32 pixels over [-2.5, 2.5]^2
EPOCHS, BATCH, LR = 15, 128, 1e-3
device = "cuda" if torch.cuda.is_available() else "cpu"
print("python", platform.python_version(), "| numpy", np.__version__, "| sklearn", sklearn.__version__,
      "| torch", torch.__version__, "| device:", device)

## 1. Scenario, signal model and data generation

A receiver gets a burst of symbols of unknown modulation (one of 4) and must classify it. Symbol timing is assumed recovered (1 sample per symbol).

**Signal model** for symbol *n* of one burst:

`r[n] = h * exp(j(2*pi*f*n + phi)) * s[n] + w[n]`

* `s[n]`: unit-power BPSK / QPSK / 16-QAM / 64-QAM symbol
* `h`: flat block-fading gain, constant within a burst. AWGN: `h = 1`. Rayleigh: `h ~ CN(0,1)`. Rician (K = 6 dB): LoS + scattered. `E|h|^2 = 1` so the SNR is the *average* SNR.
* `phi ~ U(0, 2*pi)`: unknown carrier phase. `f`: carrier-frequency offset (CFO, cycles/symbol), 0 in training.
* `w[n] ~ CN(0, 10^(-SNR/10))`: noise.
* The receiver applies AGC (normalises every window to unit power) before classification.

**Training data:** AWGN + Rayleigh, SNR -5 to 20 dB, no CFO. **Unseen test conditions:** Rician fading, SNR below/above the training range, and CFO.

In [ ]:
def constellation(mod):
    if mod == "BPSK": pts = np.array([-1, 1], complex)
    elif mod == "QPSK": pts = np.array([1+1j, 1-1j, -1+1j, -1-1j])
    else:
        m = 4 if mod == "16-QAM" else 8
        lv = np.arange(-(m-1), m, 2); pts = (lv[:, None] + 1j*lv[None, :]).ravel()
    return pts / np.sqrt(np.mean(np.abs(pts)**2))
CONST = {m: constellation(m) for m in MODS}

def cn(shape, rng):
    return (rng.standard_normal(shape) + 1j*rng.standard_normal(shape)) / np.sqrt(2)

def generate(channel, snrs, n_per_cell, cfo_max, rng):
    """Bursts of N_SYM*WIN_PER_BURST symbols, cut into WIN_PER_BURST windows. Windows of one burst share
    the same channel gain, phase, CFO and SNR, so they must never be split across train and test."""
    L = N_SYM*WIN_PER_BURST
    n_idx = np.arange(L)
    R, y, snr_l, burst, b0 = [], [], [], [], 0
    for mi, mod in enumerate(MODS):
        pts = CONST[mod]
        for snr in snrs:
            n = n_per_cell
            s = pts[rng.integers(0, len(pts), (n, L))]
            if channel == "awgn":
                h = np.ones(n, complex)
            elif channel == "rayleigh":
                h = cn((n,), rng)
            elif channel == "rician":
                k = 10**(RICIAN_K_DB/10)
                h = np.sqrt(k/(k+1))*np.exp(1j*rng.uniform(0, 2*np.pi, n)) + np.sqrt(1/(k+1))*cn((n,), rng)
            else:
                raise ValueError(channel)
            phi = rng.uniform(0, 2*np.pi, n)
            f = rng.uniform(-cfo_max, cfo_max, n)
            r = (h*np.exp(1j*phi))[:, None]*np.exp(2j*np.pi*f[:, None]*n_idx)*s + 10**(-snr/20)*cn((n, L), rng)
            r = r.reshape(n*WIN_PER_BURST, N_SYM)
            r = r/np.sqrt(np.mean(np.abs(r)**2, axis=1, keepdims=True))        # AGC per window
            R.append(r.astype(np.complex64)); y += [mi]*(n*WIN_PER_BURST); snr_l += [snr]*(n*WIN_PER_BURST)
            burst.append(np.repeat(np.arange(b0, b0+n), WIN_PER_BURST)); b0 += n
    return {"r": np.concatenate(R), "y": np.array(y), "snr": np.array(snr_l), "burst": np.concatenate(burst)}

def generate_pool(snrs, n_per_cell, rng):
    parts = [generate(ch, snrs, n_per_cell, 0.0, rng) for ch in TRAIN_CHANNELS]
    off = 0
    for p in parts:
        p["burst"] = p["burst"] + off; off = p["burst"].max() + 1
    return {k: np.concatenate([p[k] for p in parts]) for k in parts[0]}

rng = np.random.default_rng(0)
demo = generate("awgn", [20, 5], 1, 0.0, rng)
fig, ax = plt.subplots(2, 4, figsize=(12, 6))
for row, snr in enumerate([20, 5]):
    for mi, mod in enumerate(MODS):
        i = np.where((demo["snr"] == snr) & (demo["y"] == mi))[0][0]
        ax[row, mi].scatter(demo["r"][i].real, demo["r"][i].imag, s=6)
        ax[row, mi].set(title=f"{mod}, {snr} dB (AWGN)", xlim=(-2.5, 2.5), ylim=(-2.5, 2.5)); ax[row, mi].set_aspect("equal")
plt.tight_layout(); plt.show()

## 2. Baseline: handcrafted features + random forest

Handcrafted features are classic higher-order cumulants. They are chosen because they ignore the unknown carrier phase and separate the modulation orders:

* `|C20|`, `|C40|`, `|C41|`, `C42` (normalised cumulants), `E|r|^4`, `E|r|^6`, and the standard deviation of `|r|`.

In [ ]:
def features(r):
    r = r.astype(np.complex128)
    a = np.abs(r)
    c21 = np.mean(a**2, 1)                                   # = 1 after AGC
    c20 = np.mean(r**2, 1)
    c40 = np.mean(r**4, 1) - 3*c20**2
    c41 = np.mean(r**3*np.conj(r), 1) - 3*c20*c21
    c42 = np.mean(a**4, 1) - np.abs(c20)**2 - 2*c21**2
    return np.stack([np.abs(c20), np.abs(c40), np.abs(c41), c42.real,
                     np.mean(a**4, 1), np.mean(a**6, 1), np.std(a, 1)], axis=1)

def train_baseline(Ftr, ytr, seed):
    return RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1).fit(Ftr, ytr)

## 3. ML method: small CNN on constellation images

Each window is turned into a 32x32 constellation image (a 2-D histogram of the received I/Q points, shown as `log(1+count)`). The CNN has three small conv blocks and about 35k parameters. A CNN is a natural choice because the information is the *shape* of the point cloud.

In [ ]:
def to_images(r):
    ix = np.clip(((r.real + IMG_RANGE)/(2*IMG_RANGE)*IMG).astype(int), 0, IMG-1)
    iy = np.clip(((r.imag + IMG_RANGE)/(2*IMG_RANGE)*IMG).astype(int), 0, IMG-1)
    flat = (ix*IMG + iy) + np.arange(len(r))[:, None]*IMG*IMG
    counts = np.bincount(flat.ravel(), minlength=len(r)*IMG*IMG).reshape(len(r), 1, IMG, IMG)
    return np.log1p(counts).astype(np.float32)

class SmallCNN(nn.Module):
    def __init__(self, n_cls=4):
        super().__init__()
        self.f = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                               nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                               nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.c = nn.Sequential(nn.Flatten(), nn.Linear(32*(IMG//8)**2, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, n_cls))
    def forward(self, x): return self.c(self.f(x))

@torch.no_grad()
def cnn_predict(model, imgs, bs=1024):
    model.eval(); out = []
    for i in range(0, len(imgs), bs):
        out.append(model(torch.tensor(imgs[i:i+bs]).to(device)).argmax(1).cpu().numpy())
    return np.concatenate(out)

def train_cnn(Itr, ytr, Iva, yva, seed):
    torch.manual_seed(seed)
    model = SmallCNN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR); loss_fn = nn.CrossEntropyLoss()
    Xt, yt = torch.tensor(Itr), torch.tensor(ytr)
    g = torch.Generator().manual_seed(seed)
    best, best_state, hist = -1, None, []
    for ep in range(EPOCHS):
        model.train(); perm = torch.randperm(len(Xt), generator=g); run = 0.0
        for i in range(0, len(Xt), BATCH):
            idx = perm[i:i+BATCH]
            opt.zero_grad(); loss = loss_fn(model(Xt[idx].to(device)), yt[idx].to(device)); loss.backward(); opt.step()
            run += loss.item()*len(idx)
        va = np.mean(cnn_predict(model, Iva) == yva)            # model selection on validation BURSTS
        hist.append((run/len(Xt), va))
        if va > best: best, best_state = va, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, np.array(hist)

print("CNN parameters:", sum(p.numel() for p in SmallCNN().parameters()))

## 4. Leakage-safe split

The 4 windows of a burst share the same channel gain, phase and SNR, so they are near-duplicates. The pool is split **by burst** (60/20/20), the code asserts that no burst appears in two splits, and the CNN's best epoch is chosen on validation bursts only. All robustness test sets (Rician, extra SNRs, CFO) are generated fresh with a separate random stream, so they never touch training or validation.

In [ ]:
def make_splits(burst, rng, mode="burst"):
    nb = burst.max() + 1
    if mode == "burst":
        perm = rng.permutation(nb); ntr, nva = int(.6*nb), int(.2*nb)
        grp = np.empty(nb, int); grp[perm[:ntr]] = 0; grp[perm[ntr:ntr+nva]] = 1; grp[perm[ntr+nva:]] = 2
        g = grp[burst]
    else:                                                      # random window split (leaky, demo only)
        g = rng.choice([0, 1, 2], size=len(burst), p=[.6, .2, .2])
    return g == 0, g == 1, g == 2

def check_no_overlap(burst, tr, va, te):
    a, b, c = set(burst[tr]), set(burst[va]), set(burst[te])
    assert not (a & b) and not (a & c) and not (b & c), "burst leakage!"

## 5. Run the experiment (3 seeds)

In [ ]:
BASE, CNNN = "Baseline: cumulants + RF", "Proposed: CNN"
rows, conf, hists, timing = [], {}, [], {}
t_all = time.time()

def evaluate_set(d, rf, cnn, label, extra, keep_conf=False):
    F = features(d["r"]); I = to_images(d["r"])
    preds = {BASE: rf.predict(F), CNNN: cnn_predict(cnn, I)}
    for snr in np.unique(d["snr"]):
        m = d["snr"] == snr
        for name, p in preds.items():
            rows.append({"seed": seed, "method": name, "condition": label, "snr": snr, **extra,
                         "acc": np.mean(p[m] == d["y"][m])})
            if keep_conf and snr in (0, 10, 20):
                conf.setdefault((name, int(snr)), np.zeros((4, 4), int))
                conf[(name, int(snr))] += confusion_matrix(d["y"][m], p[m], labels=range(4))

for seed in SEEDS:
    rng = np.random.default_rng(seed)
    pool = generate_pool(SNR_TRAIN, BURSTS_PER_CELL, rng)
    tr, va, te = make_splits(pool["burst"], rng, "burst"); check_no_overlap(pool["burst"], tr, va, te)
    F, I, y = features(pool["r"]), to_images(pool["r"]), pool["y"]
    rf = train_baseline(F[tr], y[tr], seed)
    cnn, h = train_cnn(I[tr], y[tr], I[va], y[va], seed); hists.append(h)

    # (a) held-out bursts from the training distribution
    for name, p in {BASE: rf.predict(F[te]), CNNN: cnn_predict(cnn, I[te])}.items():
        rows.append({"seed": seed, "method": name, "condition": "In-distribution test", "snr": np.nan, "cfo": 0.0, "acc": np.mean(p == y[te])})

    # (b) fresh test sets: never used for training or model selection
    trng = np.random.default_rng(10_000 + seed)
    for ch, label in [("awgn", "AWGN"), ("rayleigh", "Rayleigh"), ("rician", "Rician (unseen)")]:
        d = generate(ch, SNR_TEST, TEST_BURSTS_PER_CELL, 0.0, trng)
        evaluate_set(d, rf, cnn, label, {"cfo": 0.0}, keep_conf=(ch == "awgn"))
    for cfo in CFO_TEST:
        d = generate("awgn", [CFO_TEST_SNR], 200, cfo, trng)
        evaluate_set(d, rf, cnn, "CFO sweep (AWGN, 15 dB)", {"cfo": cfo})

    if seed == SEEDS[0]:                                       # cost metrics on the first seed
        d = generate("awgn", [10], 100, 0.0, trng); n = len(d["y"])
        t0 = time.time(); rf.predict(features(d["r"])); timing[BASE] = (time.time()-t0)/n*1e3
        t0 = time.time(); cnn_predict(cnn, to_images(d["r"])); timing[CNNN] = (time.time()-t0)/n*1e3
    print(f"seed {seed} done | elapsed {time.time()-t_all:.0f}s")

res = pd.DataFrame(rows)

## 6. Results: tables, SNR curves, confusion matrices, CFO sweep

In [ ]:
def ms(x): return f"{100*x.mean():.1f} ± {100*x.std():.1f}"
picks = [("In-distribution test", None), ("AWGN", 0), ("AWGN", 10), ("AWGN", 20), ("Rayleigh", 10), ("Rayleigh", 20),
         ("Rician (unseen)", 10), ("Rician (unseen)", 20), ("AWGN", -10), ("AWGN", 30)]
tab = {}
for cond, snr in picks:
    sub = res[res.condition == cond] if snr is None else res[(res.condition == cond) & (res.snr == snr)]
    name = cond if snr is None else f"{cond}, {snr} dB"
    tab[name] = {m: ms(sub[sub.method == m].acc) for m in (BASE, CNNN)}
cfo_rows = res[res.condition.str.startswith("CFO")]
for cfo in CFO_TEST:
    s = cfo_rows[cfo_rows.cfo == cfo]; tab[f"AWGN 15 dB, CFO max {cfo}"] = {m: ms(s[s.method == m].acc) for m in (BASE, CNNN)}
table = pd.DataFrame(tab).T; table.columns = ["Baseline: cumulants + RF (acc %)", "Proposed: CNN (acc %)"]
display(table); table.to_csv("results_main_table.csv")
print(f"\nCost (per window): baseline {timing[BASE]:.4f} ms, CNN {timing[CNNN]:.4f} ms | CNN parameters: "
      f"{sum(p.numel() for p in SmallCNN().parameters()):,} (baseline: 200 trees)")

In [ ]:
# ---- Accuracy vs SNR (training range shaded) ----
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5), sharey=True)
for a, cond in zip(ax, ["AWGN", "Rayleigh", "Rician (unseen)"]):
    for m, c in [(BASE, "tab:orange"), (CNNN, "tab:blue")]:
        g = res[(res.condition == cond) & (res.method == m)].groupby("snr").acc.agg(["mean", "std"])
        a.plot(g.index, 100*g["mean"], "o-", color=c, label=m); a.fill_between(g.index, 100*(g["mean"]-g["std"]), 100*(g["mean"]+g["std"]), color=c, alpha=.2)
    a.axvspan(SNR_TRAIN.min(), SNR_TRAIN.max(), color="gray", alpha=.1); a.axhline(25, color="k", ls=":", lw=.8)
    a.set(xlabel="SNR (dB)", title=cond + (" - not seen in training" if "unseen" in cond else "")); a.grid(alpha=.3)
ax[0].set_ylabel("Accuracy (%)"); ax[0].legend(); plt.suptitle("Accuracy vs SNR (gray = training SNR range, dotted = chance level)")
plt.tight_layout(); plt.savefig("fig_accuracy_vs_snr.png", dpi=150); plt.show()

# ---- Confusion matrices at low / medium / high SNR (AWGN, summed over seeds) ----
fig, ax = plt.subplots(2, 3, figsize=(13, 8))
for r_i, name in enumerate([BASE, CNNN]):
    for c_i, snr in enumerate([0, 10, 20]):
        cm = conf[(name, snr)]; cmn = cm/cm.sum(1, keepdims=True)
        a = ax[r_i, c_i]; a.imshow(cmn, vmin=0, vmax=1, cmap="Blues")
        a.set(xticks=range(4), yticks=range(4), xticklabels=MODS, yticklabels=MODS, title=f"{name.split(':')[0]}, {snr} dB",
              xlabel="Predicted", ylabel="True")
        for i in range(4):
            for j in range(4): a.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", color="white" if cmn[i, j] > .5 else "black")
plt.tight_layout(); plt.savefig("fig_confusion.png", dpi=150); plt.show()

# ---- Robustness to carrier-frequency offset ----
fig, ax = plt.subplots(figsize=(6, 4.5))
for m, c in [(BASE, "tab:orange"), (CNNN, "tab:blue")]:
    g = cfo_rows[cfo_rows.method == m].groupby("cfo").acc.agg(["mean", "std"])
    ax.errorbar(g.index, 100*g["mean"], 100*g["std"], fmt="o-", color=c, label=m)
ax.set(xlabel="max CFO (cycles/symbol)", ylabel="Accuracy (%)", title="Robustness to CFO (AWGN, 15 dB, no CFO in training)"); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig("fig_cfo.png", dpi=150); plt.show()

## 7. Leakage check: random window split vs burst split

A random window split puts windows of the same burst (same channel gain, phase and SNR) in both training and test sets. The test windows then have near-duplicates in training. This section trains both methods under both splits on the same data.

In [ ]:
rng = np.random.default_rng(0)
pool = generate_pool(SNR_TRAIN, BURSTS_PER_CELL, rng)
F, I, y = features(pool["r"]), to_images(pool["r"]), pool["y"]
leak = []
for mode in ["burst", "random"]:
    tr, va, te = make_splits(pool["burst"], np.random.default_rng(0), mode)
    shared = len(set(pool["burst"][tr]) & set(pool["burst"][te]))
    rf = train_baseline(F[tr], y[tr], 0); cnn, _ = train_cnn(I[tr], y[tr], I[va], y[va], 0)
    leak.append({"split": mode, "bursts shared by train and test": shared,
                 "Baseline acc (%)": round(100*np.mean(rf.predict(F[te]) == y[te]), 1),
                 "CNN acc (%)": round(100*np.mean(cnn_predict(cnn, I[te]) == y[te]), 1)})
leak = pd.DataFrame(leak).set_index("split"); display(leak)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(2); w = 0.38
ax[0].bar(x - w/2, leak["Baseline acc (%)"], w, label="Baseline"); ax[0].bar(x + w/2, leak["CNN acc (%)"], w, label="CNN")
ax[0].set_xticks(x); ax[0].set_xticklabels(leak.index); ax[0].set(ylabel="Test accuracy (%)", title="Burst split vs random (leaky) split")
ax[0].set_ylim(50, 100); ax[0].legend(); ax[0].grid(alpha=.3, axis="y")
h = np.mean(hists, axis=0); ax[1].plot(h[:, 1]*100, label="validation acc (held-out bursts)"); ax[1].set(xlabel="Epoch", ylabel="%", title="CNN training (mean of seeds)")
ax2 = ax[1].twinx(); ax2.plot(h[:, 0], "C1", label="training loss"); ax2.set_ylabel("loss"); ax[1].legend(loc="center right")
plt.tight_layout(); plt.savefig("fig_leakage_training.png", dpi=150); plt.show()

## 8. How to explain the results and what to write in the report

**Read the figures with physical causes in mind (slide 117):**
* **Low SNR:** noise blurs the constellation points together, so 16-QAM and 64-QAM overlap first (confusion matrices at 0 dB). BPSK/QPSK stay separable longer.
* **Rayleigh / Rician fading:** a deep fade lowers the *effective* SNR of the whole burst, so the accuracy curve shifts toward the AWGN curve of a lower SNR. A stronger LoS (Rician) is closer to AWGN.
* **SNR outside the training range:** check the gray region of the SNR plot. A model can degrade where it was not trained.
* **CFO:** a frequency offset rotates the constellation during the window, smearing it into rings that look like a higher-order or different modulation. Handcrafted phase-invariant features are affected too because the phase changes *within* the window. This is the "synchronisation error can look like a new modulation" failure (slide 70).
* **Leakage:** Section 7 shows how much a random window split flatters the models.

**Limitations to state honestly:** simulated data only; flat block fading (no frequency selectivity); symbol timing assumed perfect; training had no CFO or phase noise; one 4-class closed set (no rejection of unknown signals); a single feature set and one CNN size were tried.

**Ideas to extend:** augment training with CFO and run a CFO estimator first; add an "unknown" class (option 4, open-set AMC); train on AWGN only and test on fading.